In [1]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [3]:
import langchain
import langchain_community

print(langchain.__version__)
print(langchain_community.__version__)

1.3.1
0.4.2


In [4]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import time
import psutil

import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

In [5]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [6]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [7]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [8]:
# ============================================================
# TEST FAISS RETRIEVAL
# ============================================================

docs = vector_store.similarity_search(

    "Harry Potter actor",

    k=2

)

print(
    "Documents Retrieved:",
    len(docs)
)

Documents Retrieved: 2


In [9]:
# ============================================================
# LOAD LLAMA3 MODEL
# ============================================================

llm = ChatOllama(

    model="llama3",

    temperature=0

)

print(
    "Llama3 Loaded Successfully"
)


Llama3 Loaded Successfully


In [10]:
# ============================================================
# TEST LLAMA3 CONNECTION
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [11]:
# ============================================================
# EVALUATION QUERY SET
# ============================================================

queries = [

    "Harry Potter actor",

    "World Cup winner",

    "US election",

    "climate change",

    "Olympics",

    "stock market",

    "global warming",

    "terrorism",

    "education reforms",

    "covid vaccine",

    "space exploration",

    "economic crisis",

    "oil prices",

    "artificial intelligence",

    "healthcare policy",

    "renewable energy",

    "sports championship",

    "government budget",

    "scientific discovery",

    "international relations"
]

print(
    "Queries Loaded:",
    len(queries)
)

Queries Loaded: 20


In [12]:
# ============================================================
# RESULTS CONTAINER
# ============================================================

results = []

In [13]:
# ============================================================
# DENSE RAG PIPELINE
# ============================================================

for query in queries:

    print(f"\nProcessing Query: {query}")

    start_time = time.time()

    memory_before = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    # --------------------------------------------------------
    # RETRIEVE DOCUMENTS
    # --------------------------------------------------------

    docs = vector_store.similarity_search(

        query,

        k=5

    )

    context = "\n\n".join(

        [
            doc.page_content

            for doc in docs
        ]

    )

    print(
        "Retrieved Documents:",
        len(docs)
    )

    print(
        "Context Length:",
        len(context)
    )

    # --------------------------------------------------------
    # CREATE PROMPT
    # --------------------------------------------------------

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # --------------------------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------------------------

    response = llm.invoke(
        prompt
    )

    answer = response.content

    latency = (

        time.time()

        -

        start_time

    )

    memory_after = (

        psutil.Process()
        .memory_info()
        .rss

        /

        1024

        /

        1024

    )

    memory_used = (

        memory_after

        -

        memory_before

    )

    # --------------------------------------------------------
    # SAVE RESULTS
    # --------------------------------------------------------

    results.append({

        "pipeline":
        "Dense Llama3",

        "retrieval_method":
        "Dense",

        "model":
        "Llama3",

        "query":
        query,

        "retrieved_count":
        len(docs),

        "retrieved_docs":
        str(
            [
                doc.page_content[:500]

                for doc in docs
            ]
        ),

        "context":
        context,

        "context_length":
        len(context),

        "generated_answer":
        answer,

        "response_length":
        len(answer),

        "latency_seconds":
        latency,

        "memory_mb":
        memory_used

    })

print(
    "\nDense Llama3 Pipeline Completed"
)


Processing Query: Harry Potter actor
Retrieved Documents: 5
Context Length: 2160

Processing Query: World Cup winner
Retrieved Documents: 5
Context Length: 2002

Processing Query: US election
Retrieved Documents: 5
Context Length: 2494

Processing Query: climate change
Retrieved Documents: 5
Context Length: 2043

Processing Query: Olympics
Retrieved Documents: 5
Context Length: 2483

Processing Query: stock market
Retrieved Documents: 5
Context Length: 2498

Processing Query: global warming
Retrieved Documents: 5
Context Length: 2486

Processing Query: terrorism
Retrieved Documents: 5
Context Length: 1990

Processing Query: education reforms
Retrieved Documents: 5
Context Length: 1473

Processing Query: covid vaccine
Retrieved Documents: 5
Context Length: 2057

Processing Query: space exploration
Retrieved Documents: 5
Context Length: 1960

Processing Query: economic crisis
Retrieved Documents: 5
Context Length: 2077

Processing Query: oil prices
Retrieved Documents: 5
Context Length:

In [14]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 20


,pipeline,retrieval_method,model,query,retrieved_count,retrieved_docs,context,context_length,generated_answer,response_length,latency_seconds,memory_mb
0,Dense Llama3,Dense,Llama3,Harry Potter actor,5,['malfoy rupert grint ron weasley and director...,malfoy rupert grint ron weasley and director d...,2160,"Based on the provided context, I would answer ...",103,20.234600,941.312500
1,Dense Llama3,Dense,Llama3,World Cup winner,5,['world cups including this summers tournament...,world cups including this summers tournament i...,2002,"According to the provided context, Bebeto is a...",212,15.002061,2.093750
2,Dense Llama3,Dense,Llama3,US election,5,['president since 1964 going into the election...,president since 1964 going into the election n...,2494,The 2008 US presidential election! According t...,739,32.899846,-272.953125
3,Dense Llama3,Dense,Llama3,climate change,5,['climate change let us know in the sound off ...,climate change let us know in the sound off bo...,2043,"Based on the provided context, here's what I c...",651,24.461826,253.078125
4,Dense Llama3,Dense,Llama3,Olympics,5,['one another thats what the olympics are abou...,one another thats what the olympics are about ...,2483,"According to the provided context, the Olympic...",339,19.919573,-4.093750


In [15]:
# ============================================================
# CHECK DATAFRAME STRUCTURE
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   pipeline          20 non-null     str    
 1   retrieval_method  20 non-null     str    
 2   model             20 non-null     str    
 3   query             20 non-null     str    
 4   retrieved_count   20 non-null     int64  
 5   retrieved_docs    20 non-null     str    
 6   context           20 non-null     str    
 7   context_length    20 non-null     int64  
 8   generated_answer  20 non-null     str    
 9   response_length   20 non-null     int64  
 10  latency_seconds   20 non-null     float64
 11  memory_mb         20 non-null     float64
dtypes: float64(2), int64(3), str(7)
memory usage: 100.4 KB


In [16]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "dense_llama3_results.csv",

    index=False

)

print(
    "Dense Llama3 Results Saved Successfully"
)

Dense Llama3 Results Saved Successfully
